**Calculating repetitions and false starts**
This code captures the following cases of repetitions:
1: words that are immediately repeated
2: words repeated in a span of 3 tokens
3: 2grams and 3grams that are repeated in a span of 3 tokens
4: partial words or words followed by -, not appearing at the end of a segment, or repeated syllables within a word.

In [ ]:
import re
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
from google.colab import drive, files
file_path = "/content/drive/MyDrive/NomadLingoDisfluencies/F2_segments.xlsx"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_excel(file_path)
turn_id_column = df.columns[0]
turn_column = df.columns[0]

In [ ]:
repetition_patterns = {
    "Rep1": re.compile(r'\b(\w+)\s+\1\b'),  # Rep1
    "Rep2": re.compile(r'\b(\w+)\b(?:\s+\b(?!\1)\w+\b){0,2}?\s+\1\b'),  # Rep2
    "Rep3": re.compile(r'\b(\w+\s+\w+|\w+\s+\w+\s+\w+)\b(?:\s+\b(?!\1)\w+\b){0,2}?\s+\1\b'),  # Rep3
    "FalseStart": re.compile(r'\b(\w{1,3})-\1\w*\b')  # FalseStart
}
for col_name in ["Rep1", "Rep2", "Rep3", "FalseStart"]:
    df[col_name] = 0
for key, pattern in repetition_patterns.items():
    df[key] = df[turn_column].astype(str).apply(lambda x: len(pattern.findall(x)))
output_file_path = "/content/drive/MyDrive/NomadLingoDisfluencies/repetition_F2.xlsx"
df.to_excel(output_file_path, index=False)
files.download(output_file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Discourse Markers**

In [ ]:
import re
import pandas as pd
import spacy
from google.colab import drive
nlp = spacy.load("en_core_web_sm")

In [ ]:
drive.mount('/content/drive')
from google.colab import drive, files
file_path = "/content/drive/MyDrive/NomadLingoDisfluencies/F2_segments.xlsx"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_excel(file_path)
turn_column = df.columns[0]

In [ ]:
patterns = {
    "I mean": re.compile(r'\bI mean\b', re.IGNORECASE),  # Matches 'I mean'
    "Actually": re.compile(r'\bActually\b', re.IGNORECASE),  # Matches 'Actually' anywhere
    "Like": re.compile(r'\blike\b(?!\s(to|a|\b\w+ing\b))', re.IGNORECASE),  # Excludes verb/preposition use
    "You know": re.compile(r'(?<!\bdo\s)\byou know\b', re.IGNORECASE),  # Excludes 'do you know'
    "Yeah (filler)": re.compile(r'\bYeah\b', re.IGNORECASE)  # Matches all "yeah" initially
}

# Function to count occurrences of 'So' NOT preceding an adjective and 'Well' NOT following a verb
def count_filtered_words_spacy(text, word_type):
    doc = nlp(text)
    count = 0

    for i, token in enumerate(doc):
        if word_type == "So" and token.text.lower() == "so":
            # Exclude if followed by an adjective
            if i < len(doc) - 1 and doc[i + 1].pos_ == "ADJ":
                continue
            count += 1

        elif word_type == "Well" and token.text.lower() == "well":
            # Exclude if preceded by a verb
            if i > 0 and doc[i - 1].pos_ == "VERB":
                continue
            count += 1

    return count

# Apply regex patterns to count occurrences
for key, pattern in patterns.items():
    df[key] = df["Turn"].astype(str).apply(lambda x: len(pattern.findall(str(x))))

# Apply functions to count occurrences of 'So' and 'Well' with POS filtering
df["So"] = df["Turn"].astype(str).apply(lambda x: count_filtered_words_spacy(x, "So"))
df["Well"] = df["Turn"].astype(str).apply(lambda x: count_filtered_words_spacy(x, "Well"))

# Function to count 'Really' while excluding adjectives, turn-initial position, and questions
def count_really(text):
    doc = nlp(text)
    count = 0
    for i, token in enumerate(doc):
        if token.text.lower() == "really":
            # Excludes 'Really' before adjectives
            if i < len(doc) - 1 and doc[i + 1].pos_ == "ADJ":
                continue  # Skips 'Really' if modifying an adjective
            # Excludes 'Really' at the beginning of a turn
            if i == 0:
                continue  # Skips "Really" at turn-initial position
            # Excludes "Really" before a question mark
            if i < len(doc) - 1 and doc[i + 1].text == "?":
                continue  # Skips "Really" if followed by "?"
            count += 1
    return count

df["Really"] = df["Turn"].astype(str).apply(count_really)

# Function to count 'Yeah' variants while excluding turn-initial and sequential use
def filter_yeah_variants(text):
    words = text.strip().split()
    count = 0
    for i, word in enumerate(words):
        if word.lower() in ["yeah", "yeh", "yah"]:  # Includes "Yeh" and "Yah"
            if i == 0 or (i > 0 and words[i - 1].lower() in ["yes", "yeah", "yeh", "yah"]):  # Excludes turn-initial & following "yes"/variants
                continue
            count += 1
    return count

df["Yeah (filler)"] = df["Turn"].astype(str).apply(filter_yeah_variants)

In [ ]:
output_file_path = "/content/drive/MyDrive/NomadLingoDisfluencies/F2_with_counts.xlsx"
df.to_excel(output_file_path, index=False)
files.download(output_file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>